## 데이터 파악

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/train_data.csv')
test = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/test_data.csv')
sample_submission = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/sample_submission.csv')

In [ ]:
train.head()

,index,title,topic_idx
0,0,인천→핀란드 항공기 결항…휴가철 여행객 분통,4
1,1,실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화,4
2,2,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,4
3,3,NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합,4
4,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,4


In [ ]:
train.shape

(45654, 3)

In [ ]:
test.head()

,index,title
0,45654,유튜브 내달 2일까지 크리에이터 지원 공간 운영
1,45655,어버이날 맑다가 흐려져…남부지방 옅은 황사
2,45656,내년부터 국가RD 평가 때 논문건수는 반영 않는다
3,45657,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것
4,45658,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간


In [ ]:
test.shape

(9131, 2)

In [ ]:
sample_submission.head()

,index,topic_idx
0,45654,0
1,45655,0
2,45656,0
3,45657,0
4,45658,0


In [ ]:
train.isnull().sum()

,0
index,0
title,0
topic_idx,0


In [ ]:
test.isnull().sum()

,0
index,0
title,0


In [ ]:
train.title

,title
0,인천→핀란드 항공기 결항…휴가철 여행객 분통
1,실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화
2,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것
3,NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합
4,시진핑 트럼프에 중미 무역협상 조속 타결 희망
...,...
45649,KB금융 미국 IB 스티펠과 제휴…선진국 시장 공략
45650,1보 서울시교육청 신종코로나 확산에 개학 연기·휴업 검토
45651,게시판 키움증권 2020 키움 영웅전 실전투자대회
45652,답변하는 배기동 국립중앙박물관장


In [ ]:
train.topic_idx.value_counts()

,count
topic_idx,
4,7629
2,7362
5,6933
6,6751
1,6222
3,5933
0,4824


## 전처리

### 텍스트 정제

In [ ]:
import re

def clean_text(text):
  text = re.sub(r'[“”‘’]', '', text)
  text = re.sub(r'[^가-힣A-Za-z0-9\s]', ' ', text) # 한글, 영문, 숫자, 공백만 남김
  text = re.sub(r'\s+', ' ', text).strip() # 중복공백 제거
  return text

train['cleaned'] = train['title'].apply(clean_text)
test['cleaned'] = test['title'].apply(clean_text)
print('----train----')
print(train[['title', 'cleaned']])
print('----test----')
print(test[['title', 'cleaned']])

----train----
                                    title                             cleaned
0                인천→핀란드 항공기 결항…휴가철 여행객 분통            인천 핀란드 항공기 결항 휴가철 여행객 분통
1          실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화       실리콘밸리 넘어서겠다 구글 15조원 들여 전역 거점화
2          이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것      이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것
3        NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합    NYT 클린턴 측근 기업 특수관계 조명 공과 사 맞물려종합
4               시진핑 트럼프에 중미 무역협상 조속 타결 희망           시진핑 트럼프에 중미 무역협상 조속 타결 희망
...                                   ...                                 ...
45649        KB금융 미국 IB 스티펠과 제휴…선진국 시장 공략        KB금융 미국 IB 스티펠과 제휴 선진국 시장 공략
45650     1보 서울시교육청 신종코로나 확산에 개학 연기·휴업 검토     1보 서울시교육청 신종코로나 확산에 개학 연기 휴업 검토
45651         게시판 키움증권 2020 키움 영웅전 실전투자대회         게시판 키움증권 2020 키움 영웅전 실전투자대회
45652                   답변하는 배기동 국립중앙박물관장                   답변하는 배기동 국립중앙박물관장
45653  2020 한국인터넷기자상 시상식 내달 1일 개최…특별상 김성후  2020 한국인터넷기자상 시상식 내달 1일 개최 특별상 김성후

[45654 rows x 2 columns]
----test----
           

### 토큰화

In [ ]:
!pip install kiwipiepy

In [ ]:
from kiwipiepy import Kiwi
kiwi = Kiwi()

def kiwi_tokenize(text):
    return [t.form for t in kiwi.tokenize(text)
            if t.tag.startswith(('N', 'M')) and len(t.form) > 1]

train['tokens'] = train['cleaned'].apply(kiwi_tokenize)
train['processed'] = train['tokens'].apply(lambda x: ' '.join(x))

test['tokens'] = test['cleaned'].apply(kiwi_tokenize)
test['processed'] = test['tokens'].apply(lambda x: ' '.join(x))

In [ ]:
train.head()

,index,title,topic_idx,cleaned,tokens,processed
0,0,인천→핀란드 항공기 결항…휴가철 여행객 분통,4,인천 핀란드 항공기 결항 휴가철 여행객 분통,"[인천, 핀란드, 항공기, 결항, 휴가철, 여행객, 분통]",인천 핀란드 항공기 결항 휴가철 여행객 분통
1,1,실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화,4,실리콘밸리 넘어서겠다 구글 15조원 들여 전역 거점화,"[실리콘밸리, 구글, 전역, 거점]",실리콘밸리 구글 전역 거점
2,2,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,4,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,"[이란, 외무, 긴장, 완화, 해결책, 미국, 경제, 전쟁]",이란 외무 긴장 완화 해결책 미국 경제 전쟁
3,3,NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합,4,NYT 클린턴 측근 기업 특수관계 조명 공과 사 맞물려종합,"[클린턴, 측근, 기업, 특수, 관계, 조명, 공과, 종합]",클린턴 측근 기업 특수 관계 조명 공과 종합
4,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,"[시진핑, 트럼프, 무역, 협상, 타결, 희망]",시진핑 트럼프 무역 협상 타결 희망


In [ ]:
test.head()

,index,title,cleaned,tokens,processed
0,45654,유튜브 내달 2일까지 크리에이터 지원 공간 운영,유튜브 내달 2일까지 크리에이터 지원 공간 운영,"[유튜브, 크리에이터, 지원, 공간, 운영]",유튜브 크리에이터 지원 공간 운영
1,45655,어버이날 맑다가 흐려져…남부지방 옅은 황사,어버이날 맑다가 흐려져 남부지방 옅은 황사,"[어버이날, 남부, 지방, 황사]",어버이날 남부 지방 황사
2,45656,내년부터 국가RD 평가 때 논문건수는 반영 않는다,내년부터 국가RD 평가 때 논문건수는 반영 않는다,"[내년, 국가, 평가, 논문, 건수, 반영]",내년 국가 평가 논문 건수 반영
3,45657,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,"[김명자, 신임, 과총, 회장, 원로, 과학자, 지혜]",김명자 신임 과총 회장 원로 과학자 지혜
4,45658,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,"[회색, 인간, 작가, 양심, 고백, 소설집, 출간]",회색 인간 작가 양심 고백 소설집 출간


### stopwords

In [ ]:
stopwords = set([
    '으로', '에서', '하다', '했다', '한다', '그리고', '그러나', '등',
    '대한', '관련', '위해', '통해', '대해', '것', '수', '이번', '지난', '지난해',
    '속보', '뉴스', '오늘', '내일', '기자', '보도', '사진'
])

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stopwords]

train['tokens'] = train['tokens'].apply(remove_stopwords)
train['processed'] = train['tokens'].apply(lambda x: ' '.join(x))

test['tokens'] = test['tokens'].apply(remove_stopwords)
test['processed'] = test['tokens'].apply(lambda x: ' '.join(x))

In [ ]:
train.head()

,index,title,topic_idx,cleaned,tokens,processed
0,0,인천→핀란드 항공기 결항…휴가철 여행객 분통,4,인천 핀란드 항공기 결항 휴가철 여행객 분통,"[인천, 핀란드, 항공기, 결항, 휴가철, 여행객, 분통]",인천 핀란드 항공기 결항 휴가철 여행객 분통
1,1,실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화,4,실리콘밸리 넘어서겠다 구글 15조원 들여 전역 거점화,"[실리콘밸리, 구글, 전역, 거점]",실리콘밸리 구글 전역 거점
2,2,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,4,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,"[이란, 외무, 긴장, 완화, 해결책, 미국, 경제, 전쟁]",이란 외무 긴장 완화 해결책 미국 경제 전쟁
3,3,NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합,4,NYT 클린턴 측근 기업 특수관계 조명 공과 사 맞물려종합,"[클린턴, 측근, 기업, 특수, 관계, 조명, 공과, 종합]",클린턴 측근 기업 특수 관계 조명 공과 종합
4,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,"[시진핑, 트럼프, 무역, 협상, 타결, 희망]",시진핑 트럼프 무역 협상 타결 희망


In [ ]:
test.head()

,index,title,cleaned,tokens,processed
0,45654,유튜브 내달 2일까지 크리에이터 지원 공간 운영,유튜브 내달 2일까지 크리에이터 지원 공간 운영,"[유튜브, 크리에이터, 지원, 공간, 운영]",유튜브 크리에이터 지원 공간 운영
1,45655,어버이날 맑다가 흐려져…남부지방 옅은 황사,어버이날 맑다가 흐려져 남부지방 옅은 황사,"[어버이날, 남부, 지방, 황사]",어버이날 남부 지방 황사
2,45656,내년부터 국가RD 평가 때 논문건수는 반영 않는다,내년부터 국가RD 평가 때 논문건수는 반영 않는다,"[내년, 국가, 평가, 논문, 건수, 반영]",내년 국가 평가 논문 건수 반영
3,45657,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,"[김명자, 신임, 과총, 회장, 원로, 과학자, 지혜]",김명자 신임 과총 회장 원로 과학자 지혜
4,45658,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,"[회색, 인간, 작가, 양심, 고백, 소설집, 출간]",회색 인간 작가 양심 고백 소설집 출간


### TF-IDF 벡터화

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features = 30000,
    ngram_range = (1,2),
    min_df = 2,
    max_df = 0.9,
    sublinear_tf = True,
    norm = 'l2'
)

X_train = vectorizer.fit_transform(train['processed'])
X_test = vectorizer.transform(test['processed'])
y_train = train['topic_idx']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (45654, 30000)
X_test shape: (9131, 30000)


## 모델링

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

pipeline = Pipeline([
    ('select', SelectKBest(chi2, k=15000)),  # 일단 기본값
    ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_dist = {
    'select__k': [10000, 15000, 20000],  # 피처 개수도 함께 튜닝
    'rf__n_estimators': randint(400, 1000),
    'rf__max_depth': [40, 60, 80, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__bootstrap': [True, False]
}

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='f1_macro',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('select',
                                              SelectKBest(k=15000,
                                                          score_func=<function chi2 at 0x7c859ccb5da0>)),
                                             ('rf',
                                              RandomForestClassifier(n_jobs=-1,
                                                                     random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'rf__bootstrap': [True, False],
                                        'rf__max_depth': [40, 60, 80, None],
                                        'rf__max_features': ['sqrt', 'log2'],
                                        'rf__min_samples_leaf': [1, 2, 4],
                                        'rf__min_samples_split': [2, 5, 10],
                                        'rf__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c859c82b860>,
                                        'select__k': [10000, 15000, 20000]},
                   random_state=42, scoring='f1_macro', verbose=2)

In [ ]:
# 최적 모델로 test 예측
best_model = random_search.best_estimator_
pred = best_model.predict(X_test)

sample_submission['topic_idx'] = pred
sample_submission.to_csv('/content/drive/MyDrive/data/뉴스토픽_결과_11.csv', index=False)

print("Best Parameters:", random_search.best_params_)
print("Best CV Score (f1_macro):", random_search.best_score_)
print("제출 파일 저장 완료!")

Best Parameters: {'rf__bootstrap': True, 'rf__max_depth': None, 'rf__max_features': 'log2', 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 2, 'rf__n_estimators': 960, 'select__k': 20000}
Best CV Score (f1_macro): 0.7943565788948869
제출 파일 저장 완료!
